In [ ]:
from collections import Counter
from pathlib import Path
from typing import Optional, Union

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
import scipy.spatial.distance as ssd
import seaborn as sns
from matplotlib import ticker as mtick
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgba
from matplotlib.patches import Patch, PathPatch, Wedge
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy import sparse, stats
from scipy.spatial.distance import squareform
from scipy.stats import linregress, pearsonr, spearmanr, zscore
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.decomposition import PCA

mpl.rcParams["axes.grid"] = False
mpl.rcParams["grid.alpha"] = 0.0
mpl.rcParams["grid.linewidth"] = 0.0
_TICK_RC = {
    "xtick.bottom": True,
    "ytick.left": True,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
}
mpl.rcParams.update(_TICK_RC)

def _merge_plot_rc(rc=None):
    merged = dict(_TICK_RC)
    if rc:
        merged.update(rc)
    return merged

if not hasattr(sns, "_lore_orig_set_theme"):
    sns._lore_orig_set_theme = sns.set_theme
if not hasattr(sns, "_lore_orig_set"):
    sns._lore_orig_set = sns.set

_ORIG_SNS_SET_THEME = sns._lore_orig_set_theme
_ORIG_SNS_SET = sns._lore_orig_set

def _set_theme_with_ticks(*args, **kwargs):
    kwargs["rc"] = _merge_plot_rc(kwargs.get("rc"))
    return _ORIG_SNS_SET_THEME(*args, **kwargs)

def _set_with_ticks(*args, **kwargs):
    kwargs["rc"] = _merge_plot_rc(kwargs.get("rc"))
    return _ORIG_SNS_SET(*args, **kwargs)

sns.set_theme = _set_theme_with_ticks
sns.set = _set_with_ticks
sns.set_theme(style="white", rc={"axes.grid": False, "grid.alpha": 0.0, "grid.linewidth": 0.0})

def _ensure_ticks_on_figure(fig):
    for ax in fig.get_axes():
        try:
            ax.tick_params(axis="both", which="major", direction="out", length=4, width=1)
            ax.tick_params(axis="both", which="minor", direction="out", length=2.5, width=0.8)
        except Exception:
            pass

def _disable_grids_on_figure(fig):
    for ax in fig.get_axes():
        try:
            ax.grid(False)
            ax.xaxis.grid(False, which="both")
            ax.yaxis.grid(False, which="both")
            ax.set_axisbelow(False)
        except Exception:
            pass
    _ensure_ticks_on_figure(fig)


if not hasattr(plt, "_lore_orig_show"):
    plt._lore_orig_show = plt.show

_ORIG_PLT_SHOW = plt._lore_orig_show

def _show_no_grid(*args, **kwargs):
    for fig_num in plt.get_fignums():
        _disable_grids_on_figure(plt.figure(fig_num))
    return _ORIG_PLT_SHOW(*args, **kwargs)

plt.show = _show_no_grid

def _print_donor_count(label, donor_count, *, enabled=False):
    if enabled:
        print(f"{label}: n = {int(donor_count)}")

STAGE_ORDER = [
    "Embryoblast",
    "Germ layer-specific",
    "Tissue-specific",
    "Adult-specific",
]

MUT_TYPES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]

MUT_COLORS = {
    "C>A": "#b2182b",
    "C>G": "#f781bf",
    "C>T": "#a015a5",
    "T>A": "#682aaf",
    "T>C": "#355fbb",
    "T>G": "#053061",
}

STAGE_COLORS = {
    "Embryoblast": "#FFE5CC",
    "Germ layer-specific": "#FFB366",
    "Tissue-specific": "#FF7F00",
    "Adult-specific": "#CC5500",
}

WEIGHTING_COLORS = {
    "Variant count": "#CFCFCF",
    "Supporting reads": "#4C78A8",
}


def complement(base: str) -> str:
    return {"A": "T", "T": "A", "C": "G", "G": "C"}.get(str(base).upper(), "N")


def to_pyrimidine_class(ref: str, alt: str) -> Optional[str]:
    ref = str(ref).upper()
    alt = str(alt).upper()
    if ref in {"A", "G"}:
        ref = complement(ref)
        alt = complement(alt)
    if ref in {"C", "T"} and alt in {"A", "C", "G", "T"} and ref != alt:
        return f"{ref}>{alt}"
    return None


def _locate_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data").exists() and (candidate / "Figures.ipynb").exists() and (candidate / "Supp.ipynb").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the Lore2026_Clean directory or one of its subdirectories.")


REPO_ROOT = _locate_repo_root()
TABMUR_DIR = REPO_ROOT / "data" / "TabMur"
TABSAP_DIR = REPO_ROOT / "data" / "TabSap"
TABMUR_AGG = TABMUR_DIR / "aggregates"
TABSAP_AGG = TABSAP_DIR / "aggregates"


## Fig. 1a - cumulative developmental-stage burden


In [ ]:
FIG1A_COLUMNS = ["Zygote"] + STAGE_ORDER


def _plot_cumulative_stage_burden(base_dir, age_lookup, cmap_name, age_label, dataset_label, age_unit, *, show_donor_count=False):
    df = pd.read_csv(
        Path(base_dir) / "stage_burden_per_donor_perkb_cumsum_STANDARDIZED_3LEVEL_cells.csv",
        index_col=0,
    ).reindex(columns=FIG1A_COLUMNS, fill_value=0.0)
    df = df * 1000.0

    donor_age = {}
    for donor in df.index:
        age = age_lookup(str(donor))
        if pd.notna(age):
            donor_age[str(donor)] = int(age)

    ages = sorted(set(donor_age.values()))
    cmap = plt.get_cmap(cmap_name)
    default_color = cmap(0.55)
    if len(ages) <= 1:
        norm = None
        donor_color = {str(d): default_color for d in df.index}
    else:
        norm = mpl.colors.Normalize(vmin=min(ages), vmax=max(ages))
        donor_color = {
            str(d): cmap(norm(donor_age[str(d)])) if str(d) in donor_age else default_color
            for d in df.index
        }
    donor_order = sorted(df.index, key=lambda donor: (donor_age.get(str(donor), float("inf")), str(donor)))

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1000)
    for donor in donor_order:
        row = df.loc[donor]
        ax.plot(
            df.columns,
            row[df.columns].ffill().fillna(0).to_numpy(),
            marker="o",
            linestyle="-",
            color=donor_color[str(donor)],
            alpha=0.9,
            markeredgecolor="black",
            markeredgewidth=0.3,
            linewidth=2,
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Developmental stage", fontsize=16)
    ax.set_ylabel("Cumulative mutations per megabase", fontsize=16)
    plt.setp(ax.get_xticklabels(), fontsize=14, rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), fontsize=14)

    if norm is not None:
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.04)
        cbar.set_label(age_label, fontsize=14)
        cbar.set_ticks([min(ages), max(ages)])
        cbar.ax.tick_params(labelsize=12)
    elif ages:
        ax.text(
            1.02,
            0.5,
            f"{age_label}: {ages[0]}",
            transform=ax.transAxes,
            va="center",
            fontsize=12,
        )
    plt.tight_layout()
    plt.subplots_adjust(right=0.86)
    plt.show()
    _print_donor_count(dataset_label, len(df.index), enabled=show_donor_count)
    if ages:
        print(f"{dataset_label}: age range = {min(ages)}-{max(ages)} {age_unit}")
    else:
        print(f"{dataset_label}: age range = unavailable")
    return df


mur_df = _plot_cumulative_stage_burden(
    TABMUR_AGG,
    lambda donor: pd.to_numeric(str(donor).split("-", 1)[0], errors="coerce"),
    "Blues",
    "Age (months)",
    "TabMur",
    "months",
    show_donor_count=True,
)


In [ ]:
obs = pd.read_csv(
    TABSAP_DIR / "single_cell_metadata" / "adata_obs.csv",
    usecols=["donor_id", "development_stage"],
    low_memory=False,
).dropna()
obs["age_yr"] = pd.to_numeric(obs["development_stage"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
obs = obs.dropna(subset=["age_yr"]).copy()
obs["age_yr"] = obs["age_yr"].astype(int)
donor_age = obs.groupby("donor_id")["age_yr"].agg(lambda s: int(sorted(set(s))[0])).to_dict()

sap_df = _plot_cumulative_stage_burden(
    TABSAP_AGG,
    lambda donor: donor_age.get(str(donor)),
    "Greens",
    "Age (years)",
    "TabSap",
    "years",
    show_donor_count=True,
)


## Fig. 1b - burden versus donor age and stage composition


In [ ]:
                 

FIG1B_PRIMARY_STAT = "pearson"
FIG1B_SHOW_SECONDARY_STAT = True
FIG1B_ROBUST_Z_THRESH = 4


def _sig_stars(p):
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    if p < 1e-1:
        return "#"
    return "ns"


def _summarize_iqr(df_percent):
    out = {}
    for stage in STAGE_ORDER:
        vals = df_percent[stage].dropna().to_numpy()
        out[stage] = {
            "median": np.median(vals),
            "q1": np.percentile(vals, 25),
            "q3": np.percentile(vals, 75),
        }
    return pd.DataFrame(out).T


def _load_stage_percent(base_dir):
    df = pd.read_csv(
        Path(base_dir) / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        index_col=0,
    ).reindex(columns=STAGE_ORDER, fill_value=0.0)
    return df.div(df.sum(axis=1), axis=0) * 100.0


def _load_stage_timing_donors(base_dir):
    return {str(donor) for donor in _load_stage_percent(base_dir).index}


def _robust_z(series):
    x = pd.to_numeric(series, errors="coerce")
    med = float(x.median())
    mad = float(np.median(np.abs(x - med)))
    if mad == 0.0 or np.isnan(mad):
        return pd.Series(np.nan, index=series.index, dtype=float)
    return 0.67448975 * (x - med) / mad


def _apply_robust_z_filter(df, burden_col, *, z_thresh=None, z_col=None):
    if z_thresh is None:
        return df.copy()

    out = df.copy()
    if z_col is None:
        z_col = f"{burden_col}_robust_z"

    if z_col in out.columns:
        z = pd.to_numeric(out[z_col], errors="coerce")
    else:
        z = _robust_z(out[burden_col])
        out[z_col] = z

    keep = z.isna() | (z.abs() <= z_thresh)
    return out.loc[keep].copy()


def _load_burden_summary(
    base_dir,
    summary_name,
    age_col,
    *,
    donor_whitelist=None,
    robust_z_thresh=None,
):
    df = pd.read_csv(Path(base_dir) / summary_name)
    if "min_callable_sites" in df.columns:
        df = df.sort_values(["donor", "min_callable_sites"]).drop_duplicates("donor", keep="first")
    df = _apply_robust_z_filter(
        df,
        "weighted_burden_per_kb",
        z_thresh=robust_z_thresh,
        z_col="weighted_burden_per_kb_robust_z",
    )
    df["donor"] = df["donor"].astype(str)
    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].isin(donor_whitelist)].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["donor", age_col, "weighted_burden_per_kb"])
    df["burden_per_mb"] = df["weighted_burden_per_kb"] * 1000.0
    return df


def _summarize_burden_vs_age(df, age_col):
    x = df[age_col].to_numpy(float)
    y = df["burden_per_mb"].to_numpy(float)

    stats = {
        "n": len(df),
        "n_unique_x": int(np.unique(x).size),
        "slope": np.nan,
        "intercept": np.nan,
        "pearson_r": np.nan,
        "pearson_p": np.nan,
        "spearman_rho": np.nan,
        "spearman_p": np.nan,
    }

    if np.unique(x).size > 1:
        coef = np.polyfit(x, y, 1)
        stats["slope"] = float(coef[0])
        stats["intercept"] = float(coef[1])
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        stats["pearson_r"] = float(pr)
        stats["pearson_p"] = float(pp)
        stats["spearman_rho"] = float(sr)
        stats["spearman_p"] = float(sp)

    return stats


def _format_burden_vs_age_title(stats, primary_stat="pearson", show_secondary_stat=False):
    if primary_stat == "pearson":
        primary = "Pearson r = {:.2f}, p = {:.3g} {}".format(
            stats["pearson_r"],
            stats["pearson_p"],
            _sig_stars(stats["pearson_p"]),
        )
        secondary = "Spearman rho = {:.2f}, p = {:.3g} {}".format(
            stats["spearman_rho"],
            stats["spearman_p"],
            _sig_stars(stats["spearman_p"]),
        )
    else:
        primary = "Spearman rho = {:.2f}, p = {:.3g} {}".format(
            stats["spearman_rho"],
            stats["spearman_p"],
            _sig_stars(stats["spearman_p"]),
        )
        secondary = "Pearson r = {:.2f}, p = {:.3g} {}".format(
            stats["pearson_r"],
            stats["pearson_p"],
            _sig_stars(stats["pearson_p"]),
        )
    return primary + ("\n" + secondary if show_secondary_stat else "")


def _print_burden_vs_age_stats(label, stats, age_label):
    print("\n[{}]".format(label))
    print("n = {}, unique ages = {}".format(stats["n"], stats["n_unique_x"]))
    print(
        "Linear fit: burden_per_mb = {:.12g} * {} + {:.12g}".format(
            stats["slope"],
            age_label,
            stats["intercept"],
        )
    )
    print("slope = {:.12g}".format(stats["slope"]))
    print("intercept = {:.12g}".format(stats["intercept"]))
    print(
        "Pearson r = {:.6f}, p = {:.6g}".format(
            stats["pearson_r"],
            stats["pearson_p"],
        )
    )
    print(
        "Spearman rho = {:.6f}, p = {:.6g}".format(
            stats["spearman_rho"],
            stats["spearman_p"],
        )
    )


def _plot_burden_vs_age(
    df,
    age_col,
    point_color,
    xlabel,
    *,
    stat_method="pearson",
    show_secondary_stat=True,
    show_linear_fit=True,
    show_lowess=False,
):
    x = df[age_col].to_numpy(float)
    y = df["burden_per_mb"].to_numpy(float)
    stats = _summarize_burden_vs_age(df, age_col)

    line_x = line_y = None
    lowess_curve = None
    if stats["n_unique_x"] > 1:
        if show_linear_fit:
            line_x = np.linspace(x.min(), x.max(), 200)
            line_y = np.polyval([stats["slope"], stats["intercept"]], line_x)
        if show_lowess and len(df) >= 4:
            lowess_curve = lowess(y, x, frac=0.7, return_sorted=True)

    fig, ax = plt.subplots(figsize=(12, 6), dpi=1000)
    ax.scatter(df[age_col], df["burden_per_mb"], color=point_color, edgecolor="black", s=60, linewidth=0.3)
    if line_x is not None:
        ax.plot(line_x, line_y, color="black", linestyle="-", linewidth=1.6)
    if lowess_curve is not None:
        ax.plot(lowess_curve[:, 0], lowess_curve[:, 1], color="#444444", linewidth=2.2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("Mutation burden per megabase", fontsize=16)
    ax.tick_params(axis="both", which="major", direction="out", length=6, width=1, bottom=True, left=True, top=False, right=False)
    plt.setp(ax.get_xticklabels(), fontsize=14)
    plt.setp(ax.get_yticklabels(), fontsize=14)
    ax.set_title(_format_burden_vs_age_title(stats, stat_method, show_secondary_stat))
    plt.tight_layout()
    plt.subplots_adjust(right=0.8)
    plt.show()
    return fig, ax, stats


def _average_stage_percent(base_dir):
    return _load_stage_percent(base_dir).mean().reindex(STAGE_ORDER).round(2)


def _show_stage_pies(stage_series_a, stage_series_b, labels, titles, donor_counts=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)
    for ax, avg, title in zip(axes, [stage_series_a, stage_series_b], titles):
        ax.pie(
            avg.values,
            colors=[STAGE_COLORS[s] for s in avg.index],
            startangle=90,
            wedgeprops={"edgecolor": "black", "linewidth": 0.5},
        )
        ax.set_title(title)
        ax.axis("equal")
    fig.legend(labels, title="Stage", bbox_to_anchor=(1.05, 0.5), loc="center left", frameon=False)
    plt.tight_layout()
    plt.show()
    if donor_counts is not None:
        for title, donor_count in zip(titles, donor_counts):
            _print_donor_count(title, donor_count)


def _load_age_specific_pie(base_dir, slope, intercept, tmax):
    base_stages = list(STAGE_ORDER)
    all_stages = base_stages + ["Age-specific"]
    colors = {
        "Embryoblast": "#FFE3C2",
        "Germ layer-specific": "#FFB066",
        "Tissue-specific": "#FF7A1A",
        "Adult-specific": "#CC4D00",
        "Age-specific": "#5A5A5A",
    }

    df = pd.read_csv(
        Path(base_dir) / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        index_col=0,
    ).reindex(columns=base_stages, fill_value=0.0)

    tri = 0.5 * slope * tmax ** 2
    rect = intercept * tmax
    raw_age_fraction = tri / (tri + rect)
    age_fraction = max(0.0, min(1.0, raw_age_fraction))
    if not np.isfinite(raw_age_fraction) or abs(raw_age_fraction - age_fraction) > 1e-12:
        print(
            "[WARN] Age-specific fraction was clamped from {:.4f} to {:.4f}; check whether the slope/intercept model is appropriate.".format(
                raw_age_fraction,
                age_fraction,
            )
        )

    expanded = df.copy()
    adult_vals = expanded["Adult-specific"].copy()
    expanded["Adult-specific"] = (1 - age_fraction) * adult_vals
    expanded["Age-specific"] = age_fraction * adult_vals
    avg = (expanded.div(expanded.sum(axis=1), axis=0) * 100.0).mean().reindex(all_stages).round(2)
    return avg, all_stages, colors





In [ ]:

df = _load_burden_summary(
    TABMUR_AGG,
    "tabmur_burden_vs_age__donor_summaries.csv",
    "age_mo",
    robust_z_thresh=FIG1B_ROBUST_Z_THRESH,
)

stats = _summarize_burden_vs_age(df, "age_mo")
_print_burden_vs_age_stats("Mouse (TabMur)", stats, "age_mo")
_plot_burden_vs_age(
    df,
    "age_mo",
    "#1F78B4",
    "Donor age (months)",
    stat_method=FIG1B_PRIMARY_STAT,
    show_secondary_stat=FIG1B_SHOW_SECONDARY_STAT,
    show_linear_fit=True,
    show_lowess=False,
)


In [ ]:

df = _load_burden_summary(
    TABSAP_AGG,
    "tabsap_burden_vs_age__donor_summaries.csv",
    "age_yr",
    robust_z_thresh=FIG1B_ROBUST_Z_THRESH,
)

stats = _summarize_burden_vs_age(df, "age_yr")
_print_burden_vs_age_stats("Human (TabSap)", stats, "age_yr")
_plot_burden_vs_age(
    df,
    "age_yr",
    "#41AB5D",
    "Donor age (years)",
    stat_method=FIG1B_PRIMARY_STAT,
    show_secondary_stat=FIG1B_SHOW_SECONDARY_STAT,
    show_linear_fit=True,
    show_lowess=False,
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

                             
MUR_DIR = TABMUR_AGG
SAP_DIR = TABSAP_AGG

                              
STAGES = ["Embryoblast", "Germ layer-specific", "Tissue-specific", "Adult-specific"]
WEIGHT_TAG = "cells"

                                     
STAGE_COLORS = {
    "Embryoblast": "#FFE5CC",
    "Germ layer-specific": "#FFB366",
    "Tissue-specific": "#FF7F00",
    "Adult-specific": "#CC5500",
}


def load_avg_percent(base_dir, label):
    csv_path = Path(base_dir) / f"stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_{WEIGHT_TAG}.csv"
    df = pd.read_csv(csv_path, index_col=0)
    df = df[STAGES]
    percent_df = df.div(df.sum(axis=1), axis=0) * 100
    percent_df = percent_df.round(2)
    avg = percent_df.mean().round(2)
    print(f"\n=== {label}: Average Percent Contribution ===")
    for stage in STAGES:
        print(f"{stage:20s}: {avg[stage]}%")
    return avg, len(percent_df.index)


avg_mur, mur_donor_count = load_avg_percent(MUR_DIR, "Mouse (TabMur)")
avg_sap, sap_donor_count = load_avg_percent(SAP_DIR, "Human (TabSap)")

fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)
mur_colors = [STAGE_COLORS[s] for s in avg_mur.index]
sap_colors = [STAGE_COLORS[s] for s in avg_sap.index]

axes[0].pie(
    avg_mur.values,
    colors=mur_colors,
    startangle=90,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5},
)
axes[0].set_title("Mouse (TabMur)\nAverage Across Donors")
axes[0].axis("equal")

axes[1].pie(
    avg_sap.values,
    colors=sap_colors,
    startangle=90,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5},
)
axes[1].set_title("Human (TabSap)\nAverage Across Donors")
axes[1].axis("equal")

fig.legend(
    STAGES,
    title="Stage",
    bbox_to_anchor=(1.05, 0.5),
    loc="center left",
    frameon=False,
)

plt.tight_layout()
plt.show()
_print_donor_count("Mouse (TabMur) Average Across Donors", mur_donor_count)
_print_donor_count("Human (TabSap) Average Across Donors", sap_donor_count)



In [ ]:
mur_fit_df = _load_burden_summary(
    TABMUR_AGG,
    "tabmur_burden_vs_age__donor_summaries.csv",
    "age_mo",
    robust_z_thresh=FIG1B_ROBUST_Z_THRESH,
)
mur_stats = _summarize_burden_vs_age(mur_fit_df, "age_mo")

sap_fit_df = _load_burden_summary(
    TABSAP_AGG,
    "tabsap_burden_vs_age__donor_summaries.csv",
    "age_yr",
    robust_z_thresh=FIG1B_ROBUST_Z_THRESH,
)
sap_stats = _summarize_burden_vs_age(sap_fit_df, "age_yr")

MUR_SLOPE = mur_stats["slope"]
MUR_INTERCEPT = mur_stats["intercept"]
SAP_SLOPE = sap_stats["slope"]
SAP_INTERCEPT = sap_stats["intercept"]

print("[INFO] Mouse slope/intercept from Fig. 1B fit:", MUR_SLOPE, MUR_INTERCEPT)
print("[INFO] Human slope/intercept from Fig. 1B fit:", SAP_SLOPE, SAP_INTERCEPT)

BASE_STAGES = [
    "Embryoblast",
    "Germ layer-specific",
    "Tissue-specific",
    "Adult-specific",
]
ALL_STAGES = BASE_STAGES + ["Age-specific"]
STAGE_COLORS = {
    "Embryoblast": "#FFE5CC",
    "Germ layer-specific": "#FFB366",
    "Tissue-specific": "#FF7F00",
    "Adult-specific": "#CC5500",
    "Age-specific": "#5A5A5A",
}

def load_with_age_specific(base_dir, label, slope, intercept, tmax):
    csv_path = Path(base_dir) / f"stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_{WEIGHT_TAG}.csv"
    df = pd.read_csv(csv_path, index_col=0)
    df = df[BASE_STAGES].copy()

    tri = 0.5 * slope * (tmax ** 2)
    rect = intercept * tmax
    raw_f_age = tri / (tri + rect)
    f_age = max(0.0, min(1.0, raw_f_age))
    if not np.isfinite(raw_f_age) or abs(raw_f_age - f_age) > 1e-12:
        print(
            "[WARN] Age-specific fraction was clamped from {:.4f} to {:.4f}; check whether the slope/intercept model is appropriate.".format(
                raw_f_age,
                f_age,
            )
        )

    print(f"\n=== {label}: Lifetime Age-specific fraction using Tmax={tmax} ===")
    print(f"F_AGE = {f_age:.4f}")
    print(f"Prenatal fraction = {(1 - f_age) * 100:.2f}%")
    print(f"Postnatal fraction = {f_age * 100:.2f}%")

    adult_specific = df["Adult-specific"]
    expanded = pd.DataFrame(index=df.index)
    expanded["Embryoblast"] = df["Embryoblast"]
    expanded["Germ layer-specific"] = df["Germ layer-specific"]
    expanded["Tissue-specific"] = df["Tissue-specific"]
    expanded["Adult-specific"] = (1 - f_age) * adult_specific
    expanded["Age-specific"] = f_age * adult_specific

    percent_df = expanded.div(expanded.sum(axis=1), axis=0).mul(100.0).round(2)
    avg_percent = percent_df.mean().loc[ALL_STAGES].round(2)

    print(f"\n=== {label}: Average Percent Contribution (using Tmax) ===")
    for stage in ALL_STAGES:
        print(f"{stage:20s}: {avg_percent[stage]}%")

    return percent_df, avg_percent


mur_percent_df, avg_mur = load_with_age_specific(MUR_DIR, "Mouse (TabMur)", MUR_SLOPE, MUR_INTERCEPT, tmax=30)
sap_percent_df, avg_sap = load_with_age_specific(SAP_DIR, "Human (TabSap)", SAP_SLOPE, SAP_INTERCEPT, tmax=69)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=300)
mur_vals = [avg_mur[s] for s in ALL_STAGES]
sap_vals = [avg_sap[s] for s in ALL_STAGES]
colors = [STAGE_COLORS[s] for s in ALL_STAGES]
wedgeprops = {"edgecolor": "black", "linewidth": 0.5}

axes[0].pie(mur_vals, colors=colors, startangle=90, wedgeprops=wedgeprops)
axes[0].set_title("Mouse (TabMur)\nAverage Across Donors", fontsize=12)
axes[0].axis("equal")

axes[1].pie(sap_vals, colors=colors, startangle=90, wedgeprops=wedgeprops)
axes[1].set_title("Human (TabSap)\nAverage Across Donors", fontsize=12)
axes[1].axis("equal")

fig.legend(
    ALL_STAGES,
    title="Stage",
    bbox_to_anchor=(1.05, 0.5),
    loc="center left",
    frameon=False,
)

plt.tight_layout()
plt.show()
_print_donor_count("Mouse (TabMur) Average Across Donors", len(mur_percent_df.index))
_print_donor_count("Human (TabSap) Average Across Donors", len(sap_percent_df.index))


## Fig. 1c - mutation clonality


In [ ]:
                             


def _prepare_adult_specific_clonality(mut_csv, alias_map=None, donor_whitelist=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    if "CB" not in df.columns and "cell_barcode" in df.columns:
        df = df.rename(columns={"cell_barcode": "CB"})
    valid_stage_labels = ["Adult-specific", "Cell type-specific"]
    available_stage_labels = sorted(df["stage_label"].dropna().astype(str).unique())
    df = df[df["stage_label"].isin(valid_stage_labels)].copy()
    if df.empty:
        raise ValueError(
            f"No rows matched stage_label filter {valid_stage_labels}. "
            f"Available labels: {available_stage_labels}"
        )
    df["donor"] = df["donor"].astype(str)
    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].isin(donor_whitelist)].copy()
    df["CB"] = df["CB"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "": np.nan})
    df = df[df["CB"].notna()].copy()
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    if "var_id" in df.columns:
        df["__var_key__"] = df["var_id"].astype(str)
    elif {"#CHROM", "Start", "REF", "ALT_expected"}.issubset(df.columns):
        df["__var_key__"] = df[["#CHROM", "Start", "REF", "ALT_expected"]].astype(str).agg("|".join, axis=1)
    else:
        df["__var_key__"] = df.index.astype(str)
    return df


def _clonality_distribution(mut_csv, alias_map=None, donor_whitelist=None, max_cells=50):
    df = _prepare_adult_specific_clonality(
        mut_csv,
        alias_map=alias_map,
        donor_whitelist=donor_whitelist,
    )
    per_variant = (
        df.drop_duplicates(subset=["donor", "__var_key__", "CB"])
        .groupby(["donor", "__var_key__"], observed=True)["CB"]
        .nunique()
        .reset_index(name="n_cells")
    )
    dist = per_variant["n_cells"].value_counts().sort_index().reindex(range(1, max_cells + 1), fill_value=0)
    return dist, df["donor"].nunique()


def _plot_clonality_distribution(
    mut_csv,
    color,
    edgecolor,
    title,
    *,
    alias_map=None,
    donor_whitelist=None,
    max_cells=50,
    dpi=1500,
):
    dist, donor_count = _clonality_distribution(
        mut_csv,
        alias_map=alias_map,
        donor_whitelist=donor_whitelist,
        max_cells=max_cells,
    )
    ymax = max(dist.max(), 1)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=dpi)
    ax.bar(dist.index, dist.values, color=color, edgecolor=edgecolor, linewidth=0.6)
    ax.set_xlabel("Number of cells carrying the mutation", fontsize=16)
    ax.set_ylabel("Number of mutations (log scale)", fontsize=16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xticks(list(range(1, max_cells + 1)))
    ax.set_xticklabels([str(t) if t == 1 or t % 5 == 0 else "" for t in range(1, max_cells + 1)], fontsize=12)
    ax.set_xlim(0.5, max_cells + 0.5)
    ax.set_yscale("log")
    ax.set_ylim(1, max(10, ymax * 1.1))
    ax.tick_params(axis="both", which="major", direction="out", length=6, width=1, bottom=True, left=True, top=False, right=False)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: "{:,}".format(int(v)) if v >= 1 else "0"))
    plt.setp(ax.get_yticklabels(), fontsize=12)
    ax.set_title(title, fontsize=18)
    fig.tight_layout()
    plt.show()
    _print_donor_count(title, donor_count)
    return dist


In [ ]:
mur_clonality = _plot_clonality_distribution(
    TABMUR_AGG / "mut_table_with_stage.csv",
    "#1F78B4",
    "#0B4F6C",
    "All Tissues (TabMur) - Clonality Distribution",
    donor_whitelist=_load_stage_timing_donors(TABMUR_AGG),
)


In [ ]:
sap_clonality = _plot_clonality_distribution(
    TABSAP_AGG / "mut_table_with_stage.csv",
    "#41AB5D",
    "#006D2C",
    "All Tissues (TabSap) - Clonality Distribution",
    donor_whitelist=_load_stage_timing_donors(TABSAP_AGG),
)


## Fig. 2a - tissue-by-stage mutation burden heatmaps


In [ ]:
                 


def _load_stage_tissue_heatmap(mut_csv, cmap_name, tissue_replace=None, donor_whitelist=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].astype(str).isin(donor_whitelist)].copy()
    df = df[df["stage_label"].isin(STAGE_ORDER) & df["tissue"].notna()].copy()
    if tissue_replace:
        df["tissue"] = (
            df["tissue"].astype(str).str.replace("_", " ", regex=False).str.strip().replace(tissue_replace)
        )
    else:
        df["tissue"] = df["tissue"].astype(str).str.replace("_", " ", regex=False).str.strip()
    df["SitesPerCell"] = pd.to_numeric(df["SitesPerCell"], errors="coerce")
    df = df[df["SitesPerCell"].notna() & (df["SitesPerCell"] > 0)].copy()
    df["var_id"] = df["var_id"].astype(str)

    uniq = df.drop_duplicates(subset=["donor", "tissue", "stage_label", "var_id"])
    per_donor_mut = (
        uniq.groupby(["donor", "tissue", "stage_label"], observed=True)["var_id"]
        .nunique()
        .rename("mutations")
        .reset_index()
    )
    per_donor_exp = (
        df.groupby(["donor", "tissue", "stage_label"], observed=True)["SitesPerCell"]
        .sum()
        .rename("exposure")
        .reset_index()
    )
    per_donor = per_donor_mut.merge(per_donor_exp, on=["donor", "tissue", "stage_label"], how="outer").fillna(0)
    per_donor = per_donor[per_donor["exposure"] >= 1_000_000].copy()
    per_donor["rate_per_kb"] = (per_donor["mutations"] / per_donor["exposure"]) * 1000.0
    if per_donor.empty:
        raise ValueError("No donor x tissue x stage bins passed the exposure threshold.")
    donor_count = per_donor["donor"].nunique()

    agg = per_donor.groupby(["tissue", "stage_label"], observed=True)["rate_per_kb"].mean().reset_index()
    rate_pivot = (
        agg.pivot(index="stage_label", columns="tissue", values="rate_per_kb")
        .reindex(index=list(reversed(STAGE_ORDER)))
        .fillna(0.0)
    )
    rate_pivot = rate_pivot[rate_pivot.sum(axis=0).sort_values(ascending=False).index]

    data = rate_pivot.to_numpy()
    n_rows, n_cols = data.shape
    height = min(max(0.35 * n_rows, 4.0), 20.0)
    width = min(max(0.25 * n_cols, 6.0), 30.0)

    fig, ax = plt.subplots(figsize=(width, height), dpi=1000)
    im = ax.imshow(data, aspect="auto", cmap=cmap_name, interpolation="nearest")
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(rate_pivot.columns.tolist(), rotation=90, fontsize=10)
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(rate_pivot.index.tolist(), fontsize=10)
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(False)
    ax.tick_params(axis="both", which="both", direction="out", top=False, right=False, bottom=True, left=True)
    ax.set_xlabel("Tissue type", fontsize=12)
    ax.set_ylabel("Developmental stage", fontsize=12)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Mutations per kilobase", fontsize=12)
    cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontsize(10)

    plt.tight_layout()
    plt.show()
    _print_donor_count("Stage-by-tissue heatmap", donor_count)

                             


def _plot_tissue_mutation_spectra(mut_csv, drop_tissues=None, alias_map=None, aggregation="weighted", donor_whitelist=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].astype(str).isin(donor_whitelist)].copy()
    df = df[df["tissue"].notna()].copy()
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    alt_col = "Base_observed" if "Base_observed" in df.columns else "ALT_expected"
    df["mut_class"] = df.apply(lambda r: to_pyrimidine_class(r["REF"], r[alt_col]), axis=1)
    df = df[df["mut_class"].isin(MUT_TYPES)].copy()
    df = df[df["stage_label"].isin(STAGE_ORDER)].copy()
    if drop_tissues:
        df = df[~df["tissue"].isin(drop_tissues)].copy()

    if df.empty:
        print("[WARN] No variants remained after filtering for {}".format(mut_csv))
        return

    for stage in STAGE_ORDER:
        subdf = df[df["stage_label"] == stage].copy()
        if subdf.empty:
            continue
        donor_count = subdf["donor"].nunique()

        counts = (
            subdf.groupby(["donor", "tissue", "mut_class"], observed=True)
            .size()
            .rename("n_mut")
            .reset_index()
        )
        totals = counts.groupby(["donor", "tissue"], observed=True)["n_mut"].sum().rename("total_mut").reset_index()
        counts = counts.merge(totals, on=["donor", "tissue"], how="left")
        counts["percent"] = (counts["n_mut"] / counts["total_mut"]) * 100.0

        if aggregation == "mean":
            summary = (
                counts.groupby(["tissue", "mut_class"], observed=True)["percent"]
                .mean()
                .reset_index()
            )
        elif aggregation == "weighted":
            summary = (
                counts.assign(weighted_percent=counts["percent"] * counts["n_mut"])
                .groupby(["tissue", "mut_class"], observed=True)[["weighted_percent", "n_mut"]]
                .sum()
                .reset_index()
            )
            summary["percent"] = summary["weighted_percent"] / summary["n_mut"]
        else:
            raise ValueError("Unsupported aggregation: {}".format(aggregation))

        spectrum = (
            summary.pivot(index="tissue", columns="mut_class", values="percent")
            .reindex(columns=MUT_TYPES)
            .fillna(0.0)
            .sort_index()
        )

        fig, ax = plt.subplots(figsize=(10, 6), dpi=1200)
        bottom = np.zeros(len(spectrum), dtype=float)
        for mut_class in MUT_TYPES:
            vals = spectrum[mut_class].to_numpy(float)
            ax.bar(
                spectrum.index,
                vals,
                bottom=bottom,
                color=MUT_COLORS[mut_class],
                label=mut_class,
                edgecolor="black",
                linewidth=0.3,
            )
            bottom += vals

        ax.set_title("Stage: {}".format(stage), fontsize=14)
        ax.set_ylabel("% of Mutations", fontsize=14)
        ax.set_xlabel("Tissue", fontsize=14)
        ax.set_ylim(0, 100)
        plt.xticks(rotation=90, ha="center", fontsize=10)
        ax.tick_params(axis="y", labelsize=12)
        ax.legend(title="Mutation Type", fontsize=10, title_fontsize=12, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        plt.tight_layout()
        plt.show()
        _print_donor_count("Stage: {}".format(stage), donor_count)


In [ ]:
_load_stage_tissue_heatmap(
    TABMUR_AGG / "mut_table_with_stage.csv",
    "Blues",
    donor_whitelist=_load_stage_timing_donors(TABMUR_AGG),
)


In [ ]:
_load_stage_tissue_heatmap(
    TABSAP_AGG / "mut_table_with_stage.csv",
    "Greens",
    {
        "BoneMarrow": "Bone Marrow",
        "SalivaryGland": "Salivary Gland",
        "LymphNode": "Lymph Node",
        "SubmandibularGland": "Submandibular Gland",
        "SmallIntestine": "Small Intestine",
        "LargeIntestine": "Large Intestine",
        "SkeletalMuscle": "Skeletal Muscle",
        "PeripheralBlood": "Blood",
    },
    donor_whitelist=_load_stage_timing_donors(TABSAP_AGG),
)


## Fig. 2b - tissue clonality CDFs


In [ ]:
                 


def _make_tissue_colors(tissues):
    tissues = list(tissues)
    categorical_pool = list(sns.color_palette("tab20", 20))
    categorical_pool += list(sns.color_palette("tab20b", 20))
    categorical_pool += list(sns.color_palette("tab20c", 20))
    if len(tissues) <= len(categorical_pool):
        palette = categorical_pool[: len(tissues)]
    else:
        palette = sns.color_palette("husl", len(tissues))
    return {tissue: palette[i] for i, tissue in enumerate(tissues)}


def _plot_tissue_cdf(mut_csv, alias_map=None, donor_whitelist=None):
    df = _prepare_adult_specific_clonality(
        mut_csv,
        alias_map=alias_map,
        donor_whitelist=donor_whitelist,
    )
    donor_count = df["donor"].nunique()
    x_values = np.arange(1, 51)
    cdf_data = {}
    for tissue in sorted(df["tissue"].dropna().unique()):
        sub = df[df["tissue"] == tissue]
        sub_u = sub.drop_duplicates(subset=["donor", "__var_key__", "CB"])
        per_variant = sub_u.groupby(["donor", "__var_key__"])["CB"].nunique().reset_index(name="n_cells")
        if per_variant.empty:
            continue
        counts = per_variant["n_cells"].to_numpy()
        hist, _ = np.histogram(counts, bins=np.arange(1, 52))
        cdf_data[tissue] = np.cumsum(hist) / len(counts)

    if not cdf_data:
        raise ValueError("No tissue-specific clonality data available for CDF plot.")

    tissue_order = list(cdf_data)
    tissue_colors = _make_tissue_colors(tissue_order)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=2000)
    for tissue in tissue_order:
        cdf = cdf_data[tissue]
        ax.plot(x_values, cdf, label=tissue, linewidth=1.5, color=tissue_colors[tissue])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xscale("log")
    ax.set_xlabel("Number of cells carrying the mutation (log scale)", fontsize=16)
    ax.set_ylabel("Cumulative fraction of variants", fontsize=16)
    ax.set_xlim(1, 50)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="both", which="major", direction="out", length=6, width=1, bottom=True, left=True, top=False, right=False)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: str(int(x))))
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.legend(
        title="Tissue",
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        frameon=False,
        fontsize=10,
        title_fontsize=11,
    )
    fig.tight_layout(rect=(0, 0, 0.8, 1))
    plt.show()
    _print_donor_count("Tissue CDF", donor_count)


In [ ]:
_plot_tissue_cdf(
    TABMUR_AGG / "mut_table_with_stage.csv",
    donor_whitelist=_load_stage_timing_donors(TABMUR_AGG),
)


In [ ]:
_plot_tissue_cdf(
    TABSAP_AGG / "mut_table_with_stage.csv",
    alias_map={"Bone_Marrow": "Marrow", "Muscle": "Limb_Muscle"},
    donor_whitelist=_load_stage_timing_donors(TABSAP_AGG),
)


## Fig. 2c - mouse tissue mutation spectra


In [ ]:
_plot_tissue_mutation_spectra(
    TABMUR_AGG / "mut_table_with_stage.csv",
    alias_map={"Limb_Muscle": "Muscle"},
    aggregation="mean",
)


## Fig. 2d - human tissue mutation spectra


In [ ]:
_plot_tissue_mutation_spectra(
    TABSAP_AGG / "mut_table_with_stage.csv",
    drop_tissues={"SalivaryGland", "Prostate", "Thymus", "Trachea"},
    alias_map={"Bone_Marrow": "Marrow", "BoneMarrow": "Marrow"},
    aggregation="mean",
)


## Fig. 3a-d - mutation-expression coupling and GO enrichment


In [ ]:
                             

def _show_best_gene_example_regression_tabmur():
    data_path = TABMUR_DIR / "expression_coupling" / "example_regression_best_gene.csv.gz"
    if not data_path.exists():
        print("[WARN] Skipping TabMur example regression; missing input:")
        print("  - {}".format(data_path.relative_to(REPO_ROOT)))
        return

    df = pd.read_csv(data_path)
    if df.empty:
        print("[WARN] Skipping TabMur example regression; cached data is empty.")
        return

    best_gene = str(df["gene"].iloc[0])
    x = zscore(df["expression"].to_numpy())
    y = zscore(df["mutation_rate"].to_numpy())
    slope, intercept, r, p, stderr = linregress(x, y)

    plt.figure(figsize=(6, 5), dpi=600)
    sns.regplot(
        x=x,
        y=y,
        scatter_kws=dict(s=12, alpha=0.5, color="lightgray"),
        line_kws=dict(linewidth=2, color="black"),
    )
    plt.xlabel("{} expression (z-scored)".format(best_gene))
    plt.ylabel("Mutation rate (z-scored)")
    plt.title("TabMur example gene: r = {:.2f}, p = {:.2e}".format(r, p))
    plt.tight_layout()
    plt.show()


def _load_slopes(slopes_path):
    df = pd.read_csv(slopes_path)
    if not {"r", "p"}.issubset(df.columns):
        raise SystemExit("[ERROR] slope file missing required columns")
    df["logp"] = -np.log10(df["p"].replace(0, np.nan))
    df["r_jitter"] = df["r"] + np.random.normal(0, 1e-4, size=len(df))
    df["logp_jitter"] = df["logp"] + np.random.normal(0, 1e-2, size=len(df))
    df["sig_class"] = "ns"
    df.loc[(df["p"] <= 0.05) & (df["r"] >= 0.1), "sig_class"] = "sig_pos"
    df.loc[(df["p"] <= 0.05) & (df["r"] <= -0.1), "sig_class"] = "sig_neg"
    return df


def _plot_slope_hist(slopes_path, color, title):
    df = _load_slopes(slopes_path)
    plt.figure(figsize=(7, 6), dpi=600)
    sns.histplot(df["r"], bins=100, color=color, edgecolor=None)
    sns.despine()
    plt.xlabel("Pearson r")
    plt.ylabel("Number of genes")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def _plot_slope_volcano(slopes_path, title):
    df = _load_slopes(slopes_path)
    palette = {"sig_pos": "red", "sig_neg": "blue", "ns": "lightgray"}
    plt.figure(figsize=(7, 6), dpi=600)
    sns.scatterplot(
        data=df,
        x="r_jitter",
        y="logp_jitter",
        hue="sig_class",
        palette=palette,
        edgecolor=None,
        alpha=0.6,
        s=15,
        legend=False,
    )
    plt.xlabel("Pearson r")
    plt.ylabel("-log10(p)")
    plt.title(title)
    sns.despine()
    ymax = df["logp_jitter"].replace([np.inf, -np.inf], np.nan).dropna().max() * 1.05
    plt.ylim(0, min(ymax, 500))
    plt.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=1)
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.tight_layout()
    plt.show()
    print("Significant positive (upregulated):", int((df["sig_class"] == "sig_pos").sum()))
    print("Significant negative (downregulated):", int((df["sig_class"] == "sig_neg").sum()))
    print("Not significant:", int((df["sig_class"] == "ns").sum()))


def _load_gsea(path):
    return pd.read_csv(path, sep="\t")


def _plot_go_bar(neg_path, pos_path, title):
    df_neg = _load_gsea(neg_path)
    df_pos = _load_gsea(pos_path)
    df_neg_top = df_neg.nsmallest(10, "FDR q-val").copy()
    df_pos_top = df_pos.nsmallest(10, "FDR q-val").copy()
    df_neg_top["direction"] = "Negative"
    df_pos_top["direction"] = "Positive"
    df_plot = pd.concat([df_pos_top, df_neg_top], ignore_index=True).sort_values("NES")
    plt.figure(figsize=(12, 8), dpi=600)
    sns.barplot(
        data=df_plot,
        x="NES",
        y="NAME",
        hue="direction",
        palette={"Positive": "red", "Negative": "blue"},
    )
    sns.despine()
    plt.xlabel("Normalized Enrichment Score (NES)")
    plt.ylabel("")
    plt.title(title)
    plt.legend(title="Enrichment")
    plt.xlim(-7, 7)
    plt.tight_layout()
    plt.show()


def _collapse_terms(df, nes_col="NES", term_col="NAME"):
    tmp = df.copy()
    tmp["absNES"] = tmp[nes_col].abs()
    tmp = tmp.sort_values("absNES", ascending=False)
    tmp = tmp.drop_duplicates(subset=term_col, keep="first")
    return tmp.set_index(term_col)[nes_col]


def _plot_cross_species_go_heatmap():
    mur_neg = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
    mur_pos = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
    sap_neg = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807192566.tsv"
    sap_pos = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807192566.tsv"

    mur_pos_df = _load_gsea(mur_pos)[["NAME", "NES"]]
    mur_neg_df = _load_gsea(mur_neg)[["NAME", "NES"]]
    sap_pos_df = _load_gsea(sap_pos)[["NAME", "NES"]]
    sap_neg_df = _load_gsea(sap_neg)[["NAME", "NES"]]

    def top10(df, direction):
        return df.sort_values("NES", ascending=(direction == "down")).head(10)

    mur_up = top10(mur_pos_df, "up").assign(group="Mouse_UP")
    mur_down = top10(mur_neg_df, "down").assign(group="Mouse_DOWN")
    sap_up = top10(sap_pos_df, "up").assign(group="Human_UP")
    sap_down = top10(sap_neg_df, "down").assign(group="Human_DOWN")
    all_top = pd.concat([mur_up, mur_down, sap_up, sap_down], ignore_index=True)

    mur_nes = _collapse_terms(pd.concat([mur_pos_df, mur_neg_df], ignore_index=True))
    sap_nes = _collapse_terms(pd.concat([sap_pos_df, sap_neg_df], ignore_index=True))

    terms = all_top["NAME"]
    heat = pd.DataFrame(
        {
            "NES_mouse": mur_nes.reindex(terms),
            "NES_human": sap_nes.reindex(terms),
            "group": all_top["group"].values,
        },
        index=terms,
    )
    heat[["NES_mouse", "NES_human"]] = heat[["NES_mouse", "NES_human"]].fillna(0)
    heat["group"] = pd.Categorical(
        heat["group"],
        categories=["Human_UP", "Human_DOWN", "Mouse_UP", "Mouse_DOWN"],
        ordered=True,
    )
    heat = heat.sort_values("group", kind="stable")
    heat = heat[~heat.index.duplicated(keep="first")]

    cmap = LinearSegmentedColormap.from_list("darkblue_white_darkcoral", ["#3B6BA5", "#FFFFFF", "#D56A4A"])
    plt.figure(figsize=(13, max(6, 0.38 * heat.shape[0])))
    sns.heatmap(
        heat[["NES_mouse", "NES_human"]],
        cmap=cmap,
        center=0,
        linewidths=0,
        linecolor=None,
        cbar_kws={"label": "NES"},
    )
    plt.title("Top 10 up/down GO terms per species")
    plt.xlabel("Species")
    plt.ylabel("GO term")
    plt.tight_layout()
    plt.show()


def _plot_cross_species_go_scatter():
    mur_neg = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
    mur_pos = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
    sap_neg = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807192566.tsv"
    sap_pos = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807192566.tsv"

    mur_all = _collapse_terms(pd.concat([_load_gsea(mur_pos)[["NAME", "NES"]], _load_gsea(mur_neg)[["NAME", "NES"]]], ignore_index=True))
    sap_all = _collapse_terms(pd.concat([_load_gsea(sap_pos)[["NAME", "NES"]], _load_gsea(sap_neg)[["NAME", "NES"]]], ignore_index=True))
    overlap = sorted(set(mur_all.index) & set(sap_all.index))
    df = pd.DataFrame({"NES_mouse": mur_all.loc[overlap], "NES_human": sap_all.loc[overlap]}, index=overlap)

    def classify(row):
        if row["NES_mouse"] > 0 and row["NES_human"] > 0:
            return "Concordant up"
        if row["NES_mouse"] < 0 and row["NES_human"] < 0:
            return "Concordant down"
        return "Opposite"

    df["category"] = df.apply(classify, axis=1)
    strength = (df["NES_mouse"].abs() + df["NES_human"].abs()) / 2
    df["size"] = 20 + 60 * (strength / strength.max())

    plt.figure(figsize=(4.5, 4.5), dpi=600)
    for category, color, alpha in [
        ("Concordant up", "#D56A4A", 0.65),
        ("Concordant down", "#3B6BA5", 0.65),
        ("Opposite", "#777777", 0.40),
    ]:
        sub = df[df["category"] == category]
        plt.scatter(
            sub["NES_mouse"],
            sub["NES_human"],
            s=sub["size"] * 0.20,
            color=color,
            alpha=alpha,
            edgecolor="none",
            label=category,
        )

    plt.xlim(-7, 7)
    plt.ylim(-7, 7)
    plt.axhline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
    plt.axvline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
    plt.xlabel("NES (Mouse)", fontsize=11)
    plt.ylabel("NES (Human)", fontsize=11)
    plt.title("GO pathway enrichment across species", fontsize=12)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(False)
    plt.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.04, 0.5))
    sns.despine(top=True, right=True)
    ax = plt.gca()
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import sparse
from scipy.stats import zscore, linregress
import matplotlib.pyplot as plt
import seaborn as sns

MIN_EXPRESSING_CELLS = 10
NORMALIZE_EXPR = True
NORMALIZE_BURDEN = True
KEEP_DONORS = sorted(_load_stage_timing_donors(TABMUR_AGG))

BASE_DIR = TABMUR_DIR
SINGLE_CELL_METADATA_DIR = BASE_DIR / "single_cell_metadata"
SLOPES_CSV = BASE_DIR / "expression_coupling" / "slopes.csv"
EXPR_PATH = SINGLE_CELL_METADATA_DIR / "expression_matrix_all_genes.npz"
GENE_PATH = SINGLE_CELL_METADATA_DIR / "adata_var_names.txt"
OBS_PATH = SINGLE_CELL_METADATA_DIR / "adata_obs.csv"
MUTRATE = BASE_DIR / "expression_coupling" / "mutation_rate__celltype_specific.csv"


print(f"[INFO] Loading {SLOPES_CSV}...")
slopes = pd.read_csv(SLOPES_CSV)
slopes_filtered = slopes[slopes["n_cells_expr"] > 1000].copy()
if slopes_filtered.empty:
    raise SystemExit("[ERROR] No genes have n_cells_expr > 1000")

slopes_filtered["abs_r"] = slopes_filtered["r"].abs()
best_row = slopes_filtered.sort_values("abs_r", ascending=False).iloc[0]
BEST_GENE = best_row["gene"]
print(f"[INFO] Strongest gene with >1000 cells = {BEST_GENE}")
print(f"       r = {best_row['r']:.3f}, p = {best_row['p']:.3e}, n_cells_expr = {best_row['n_cells_expr']}")

expr = sparse.load_npz(EXPR_PATH).tocsr()
gene_names = pd.read_csv(GENE_PATH, header=None).squeeze().astype(str).values

obs = pd.read_csv(OBS_PATH, index_col=0, low_memory=False)
obs["row_idx"] = np.arange(len(obs))
obs["CB"] = obs.index.astype(str).str.split("-").str[0]
if "donor" in obs.columns:
    obs["donor"] = obs["donor"].astype(str)
elif "mouse.id" in obs.columns:
    obs["donor"] = obs["mouse.id"].astype(str)
elif "mouse_id" in obs.columns:
    obs["donor"] = obs["mouse_id"].astype(str)
else:
    raise SystemExit("[ERROR] No donor column found")
obs = obs[obs["donor"].isin(KEEP_DONORS)]

mutrate = pd.read_csv(MUTRATE)
mutrate["CB"] = mutrate["CB"].astype(str).str.split("-").str[0]
mutrate["donor"] = mutrate["donor"].astype(str)
mutrate = mutrate[mutrate["donor"].isin(KEEP_DONORS)]

merged = pd.merge(
    obs[["row_idx", "donor", "CB"]],
    mutrate[["CB", "donor", "mutation_rate"]],
    on=["CB", "donor"],
    how="left",
)
merged["mutation_rate"] = merged["mutation_rate"].fillna(0)
expr_sub = expr[merged["row_idx"].to_numpy(), :].tocsr()
merged = merged.reset_index(drop=True)

mb = merged["mutation_rate"].to_numpy()
mb_vec = zscore(mb) if NORMALIZE_BURDEN else mb

idx = np.where(gene_names == BEST_GENE)[0]
if len(idx) == 0:
    raise SystemExit(f"[ERROR] Gene {BEST_GENE} not found in expression matrix")

gi = idx[0]
expr_vals = expr_sub[:, gi].toarray().ravel()
mask = expr_vals > 0
x = expr_vals[mask]
y = mb_vec[mask]
x = zscore(x) if NORMALIZE_EXPR else x

slope, intercept, r, p, stderr = linregress(x, y)
print(f"[INFO] Regression for {BEST_GENE}:")
print(f"       n expressing cells = {mask.sum()}")
print(f"       slope = {slope:.4f}")
print(f"       r = {r:.3f}")
print(f"       p = {p:.3e}")

plt.figure(figsize=(6, 5), dpi=600)
sns.regplot(
    x=x,
    y=y,
    scatter_kws=dict(s=12, alpha=0.5, color="lightgray"),
    line_kws=dict(linewidth=2, color="black"),
)
plt.xlabel(f"{BEST_GENE} expression (z-scored)" if NORMALIZE_EXPR else f"{BEST_GENE} expression")
plt.ylabel("Mutation rate (z-scored)" if NORMALIZE_BURDEN else "Mutation rate")
plt.title(f"{BEST_GENE}: mutation–expression regression\nr = {r:.2f}, p = {p:.1e}, n = {mask.sum()}")
sns.despine()
plt.tight_layout()

plt.show()



In [ ]:
"""
Plots for TabMur (Mouse):
1. Distribution of r values
2. Volcano plot: r vs. -log10(p), styled
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
MOUSE = TABMUR_DIR / "expression_coupling" / "slopes_protein_coding.csv"

                            
print(f"[INFO] Reading {MOUSE}")
df = pd.read_csv(MOUSE)

required = {"r", "p"}
if not required.issubset(df.columns):
    raise SystemExit(f"[ERROR] Mouse file missing required columns: {required}")

                                    
df["logp"] = -np.log10(df["p"].replace(0, np.nan))
df["r_jitter"] = df["r"] + np.random.normal(0, 1e-4, size=len(df))
df["logp_jitter"] = df["logp"] + np.random.normal(0, 1e-2, size=len(df))

df["sig_class"] = "ns"
df.loc[(df["p"] <= 0.05) & (df["r"] >= 0.1), "sig_class"] = "sig_pos"
df.loc[(df["p"] <= 0.05) & (df["r"] <= -0.1), "sig_class"] = "sig_neg"

palette = {"sig_pos": "red", "sig_neg": "blue", "ns": "lightgray"}


                                    
plt.figure(figsize=(7, 6), dpi=600)
sns.scatterplot(
    data=df,
    x="r_jitter",
    y="logp_jitter",
    hue="sig_class",
    palette=palette,
    edgecolor=None,
    alpha=0.6,
    s=15,
    legend=False
)

plt.xlabel("Pearson r")
plt.ylabel("-log10(p)")
plt.title("Mice")

sns.despine()

ymax = df["logp_jitter"].replace([np.inf, -np.inf], np.nan).dropna().max() * 1.05
plt.ylim(0, min(ymax, 500))

plt.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()


n_pos = (df["sig_class"] == "sig_pos").sum()
n_neg = (df["sig_class"] == "sig_neg").sum()
n_ns  = (df["sig_class"] == "ns").sum()

print("Significant positive (upregulated):", n_pos)
print("Significant negative (downregulated):", n_neg)
print("Not significant:", n_ns)



In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

MUR_NEG = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
MUR_POS = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
SAP_NEG = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807192566.tsv"
SAP_POS = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807192566.tsv"
OUT_DIR = MUR_NEG.parent

TERM_COL = "NAME"
NES_COL = "NES"

sns.set(style="white", context="notebook", rc={"axes.grid": False, "grid.alpha": 0.0, "grid.linewidth": 0.0})

custom_cmap = LinearSegmentedColormap.from_list(
    "darkblue_white_darkcoral",
    ["#3B6BA5", "#FFFFFF", "#D56A4A"],
)


def load_gsea(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    return df[[TERM_COL, NES_COL]].copy()


def collapse_terms(df: pd.DataFrame) -> pd.Series:
    tmp = df.copy()
    tmp["absNES"] = tmp[NES_COL].abs()
    tmp = tmp.sort_values("absNES", ascending=False)
    tmp = tmp.drop_duplicates(subset=TERM_COL, keep="first")
    return tmp.set_index(TERM_COL)[NES_COL]


def top10(df: pd.DataFrame, direction: str) -> pd.DataFrame:
    if direction == "up":
        return df.sort_values(NES_COL, ascending=False).head(10)
    return df.sort_values(NES_COL, ascending=True).head(10)


def clean_go_label(term: str) -> str:
    if term.startswith("GOBP_"):
        term = term.replace("GOBP_", "", 1)
    term = term.replace("_", " ")
    return term.capitalize()


mur_pos = load_gsea(MUR_POS)
mur_neg = load_gsea(MUR_NEG)
sap_pos = load_gsea(SAP_POS)
sap_neg = load_gsea(SAP_NEG)

mur_up = top10(mur_pos, "up").assign(group="Mouse_UP")
mur_down = top10(mur_neg, "down").assign(group="Mouse_DOWN")
sap_up = top10(sap_pos, "up").assign(group="Human_UP")
sap_down = top10(sap_neg, "down").assign(group="Human_DOWN")
all_top = pd.concat([mur_up, mur_down, sap_up, sap_down], ignore_index=True)

mur_all = pd.concat([mur_pos, mur_neg], ignore_index=True)
sap_all = pd.concat([sap_pos, sap_neg], ignore_index=True)
mur_nes = collapse_terms(mur_all)
sap_nes = collapse_terms(sap_all)

terms = all_top[TERM_COL]
heat = pd.DataFrame(
    {
        "NES_mouse": mur_nes.reindex(terms),
        "NES_human": sap_nes.reindex(terms),
        "group": all_top["group"].values,
    },
    index=terms,
)

heat_filled = heat.copy()
heat_filled[["NES_mouse", "NES_human"]] = heat_filled[["NES_mouse", "NES_human"]].fillna(0)
group_order = ["Human_UP", "Human_DOWN", "Mouse_UP", "Mouse_DOWN"]
heat_filled["group"] = pd.Categorical(heat_filled["group"], categories=group_order, ordered=True)
heat_filled = heat_filled.sort_values("group", kind="stable")
heat_unique = heat_filled[~heat_filled.index.duplicated(keep="first")]

print("[INFO] Unique rows in heatmap:", heat_unique.shape[0])

plt.figure(figsize=(13, max(6, 0.38 * heat_unique.shape[0])))
sns.heatmap(
    heat_unique[["NES_mouse", "NES_human"]],
    cmap=custom_cmap,
    center=0,
    linewidths=0,
    linecolor=None,
    cbar_kws={"label": "NES"},
)
plt.title("Top 10 up/down GO terms per species (unique GO terms, missing = 0 NES)")
plt.xlabel("Species")
plt.ylabel("GO term")
plt.tight_layout()
plt.show()

cleaned_terms = [clean_go_label(term) for term in heat_unique.index]

print("\n=== Clean GO terms in heatmap order ===")
for term in cleaned_terms:
    print(term)

pd.Series(cleaned_terms).to_csv(
    OUT_DIR / "GO_top10_updown_clean_unique_noGOBP_spaces.tsv",
    sep="\t",
    index=False,
)

print(f"[✓] Saved cleaned GO list → {OUT_DIR}/GO_top10_updown_clean_unique_noGOBP_spaces.tsv")
print(f"[✓] Saved NES table → {OUT_DIR}/GO_top10_updown_mouse_human_fill0_unique.tsv")

txt_path = OUT_DIR / "GO_top10_updown_clean_unique_noGOBP_spaces.txt"
with open(txt_path, "w") as fh:
    for term in cleaned_terms:
        fh.write(term + "\n")

print(f"[✓] Saved cleaned GO-term list → {txt_path}")



## Fig. 3e - conserved pathway response summary


## Fig. 3b-e - GO concordance summary


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

MUR_NEG = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
MUR_POS = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
SAP_NEG = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807192566.tsv"
SAP_POS = TABSAP_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807192566.tsv"

TERM_COL = "NAME"
NES_COL = "NES"

sns.set(style="white", context="notebook", rc={"axes.grid": False, "grid.alpha": 0.0, "grid.linewidth": 0.0})

COLOR_CONC_UP = "#D56A4A"
COLOR_CONC_DOWN = "#3B6BA5"
COLOR_OPPOSITE = "#777777"


def load_gsea(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    if TERM_COL not in df.columns or NES_COL not in df.columns:
        raise ValueError(f"ERROR: Missing columns in {path}.")
    return df[[TERM_COL, NES_COL]].copy()


def collapse_to_one_nes(df: pd.DataFrame) -> pd.Series:
    tmp = df.copy()
    tmp["absNES"] = tmp[NES_COL].abs()
    tmp = tmp.sort_values("absNES", ascending=False)
    tmp = tmp.drop_duplicates(subset=TERM_COL, keep="first")
    return tmp.set_index(TERM_COL)[NES_COL]


mur_pos = load_gsea(MUR_POS)
mur_neg = load_gsea(MUR_NEG)
sap_pos = load_gsea(SAP_POS)
sap_neg = load_gsea(SAP_NEG)

mur_all = collapse_to_one_nes(pd.concat([mur_pos, mur_neg], ignore_index=True))
sap_all = collapse_to_one_nes(pd.concat([sap_pos, sap_neg], ignore_index=True))
overlap = sorted(set(mur_all.index) & set(sap_all.index))
print(f"[INFO] Overlapping GO terms: {len(overlap)}")

df = pd.DataFrame(
    {
        "NES_mouse": mur_all.loc[overlap],
        "NES_human": sap_all.loc[overlap],
    },
    index=overlap,
)


def classify(row):
    m = row["NES_mouse"]
    h = row["NES_human"]
    if m > 0 and h > 0:
        return "Concordant up"
    if m < 0 and h < 0:
        return "Concordant down"
    return "Opposite"


df["category"] = df.apply(classify, axis=1)
opp_df = df[df["category"] == "Opposite"]
mouse_up_human_down = ((opp_df["NES_mouse"] > 0) & (opp_df["NES_human"] < 0)).sum()
mouse_down_human_up = ((opp_df["NES_mouse"] < 0) & (opp_df["NES_human"] > 0)).sum()
df["strength"] = (df["NES_mouse"].abs() + df["NES_human"].abs()) / 2
df["size"] = 20 + 60 * (df["strength"] / df["strength"].max())

print("\n=== Overall categories ===")
print(df["category"].value_counts())
print("\n=== Opposite split ===")
print("Mouse UP / Human DOWN:", mouse_up_human_down)
print("Mouse DOWN / Human UP:", mouse_down_human_up)

plt.figure(figsize=(4.5, 4.5), dpi=600)
dot_scale = 0.20

for category, color, alpha in [
    ("Concordant up", COLOR_CONC_UP, 0.65),
    ("Concordant down", COLOR_CONC_DOWN, 0.65),
    ("Opposite", COLOR_OPPOSITE, 0.40),
]:
    sub = df[df["category"] == category]
    plt.scatter(
        sub["NES_mouse"],
        sub["NES_human"],
        s=sub["size"] * dot_scale,
        color=color,
        alpha=alpha,
        edgecolor="none",
        label=category,
    )

plt.xlim(-7, 7)
plt.ylim(-7, 7)
plt.axhline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
plt.axvline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
plt.xlabel("NES (Mouse)", fontsize=11)
plt.ylabel("NES (Human)", fontsize=11)
plt.title("GO pathway enrichment across species\n(all overlapping GO terms)", fontsize=12)
plt.gca().set_aspect("equal", adjustable="box")
plt.grid(False)
plt.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.04, 0.5))

sns.despine(top=True, right=True)
ax = plt.gca()
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

plt.tight_layout()
plt.show()

